# 리포트 6 — σ 를 무엇에 붙들어 매나

> 우리 σ 의 절대 레벨을 붙드는 것은 **공개 문헌 한 기체·한 실험실**뿐이다. 그 끈의 장력을 재고, 끊어질 자리를 먼저 적는다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현** 을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 | 만든 곳 |
|---|---|---|
| 1 ⭐ | σ = A(f)·B₁·B₂ 에서 A(f) 의 기울기만 측정에서 받고, 레벨과 각패턴은 우리 PO 출력이다 | `_parts/24_anchor-mode.ipynb` |
| 2 | 앵커가 통제한 항목과 남은 항목의 크기를 기체별 원장으로 적었다 | `_parts/25_anchor-ledger.ipynb` |
| 3 | Phantom 3 를 문헌값을 보지 않고 내고 봉인을 풀었다 | `_parts/26_blind-p3.ipynb` |
| 4 | 메쉬가 사는 축은 절대 크기가 아니라 각도 구조다 | `_parts/27_box-sphere-control.ipynb` |
| 5 | 같은 잣대를 네 기체로 넓히면 판정이 NOT_VALIDATED 로 갈린다 | `_parts/28_fleet-prereg.ipynb` |
| 6 | 공통모드 σ 오차는 파형 순위를 안 건드리고, 차분 오차가 뒤집는다 | `_parts/29_sigma-robustness.ipynb` |

⭐ 표시한 절 하나만 읽어도 이 권의 결론은 선다.

숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 절 끝 «출처» 표가 그 파일과 키다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.

전체 목차는 [reports/README.md](README.md) 이고, 열다섯 권의 지도는 [리포트 1 «이 연구가 묻는 것과 답한 방식»](01_map.ipynb) 다.


---

## 절 1. σ = A(f)·B₁·B₂ 에서 A(f) 의 기울기만 측정에서 받고, 레벨과 각패턴은 우리 PO 출력이다



> ### 한 일
> **게재된 표준트랙 논문의 분해를 그대로 쓰고, 세 인자 중 주파수 기울기 하나만 공개 측정에 정렬했다.**

### 결과
1. 기하에서 나온 우리 밴드 기울기는 -0.084 [^1] (mavic4pro [^2])~2.004 dB/GHz [^3] (phantom4 [^4]) 이고, 측정은 0.210 [^5] (Das) 와 0.315 dB/GHz [^6] (Yuan) 다.
2. 생산 기준은 `slope_only [^7]` 이고, 밴드별로 옮기는 양은 +2.70 [^8] (phantom4 [^9] @ LTE 1.843 GHz [^10]) ~ -2.41 dB [^11] (s1000plus [^12] @ WiFi 5.21 GHz [^13]) 다.
3. 정렬 후 7기종 기울기가 모두 0.210 dB/GHz [^14] 위에 서고, 기종 간 산포는 1.9e-15 dB/GHz [^15] 다.
4. 레벨이동 절대 최대는 0.00 dB [^16], 재보정 후 정규화 각패턴 변화는 1.9e-15 dB [^17] 다 — 주파수 눈금만 측정에서 오고 레벨과 모양은 그대로다.
5. 레벨까지 맞추려면 크기전이 법칙을 하나 골라야 하고, 그 선택 하나가 기체당 최대 9.50 dB [^18] 를 정한다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 분해 | σ = A(f)·B₁(φ,θ)·B₂ — 게재된 표준트랙 논문의 분해를 그대로 쓴다(Zhang, IEEE JSAC 44:702, 2026 — 측정 적합 모델) |
| 무엇을 받나 | A(f) 의 **기울기만** 측정에서 받는다. A(f) 의 레벨과 B₁ 은 우리 PO 출력이다 |
| 왜 기울기만 | 레벨까지 맞추려면 크기전이 법칙을 하나 골라야 하고, 측정이 그 대가 없이 제약하는 양은 기울기뿐이다 |
| 모드 구현 | `src/sigma_anchor.py` 가 재보정 모드 셋을 제공한다 — 생산은 `slope_only` 다 |

### 재현

```bash
PYTHONPATH=src python src/make_report02_target.py --derive-only
PYTHONPATH=src python benchmark/rcs_anchor.py
PYTHONPATH=src python src/build_part05_anchor.py
```

| | |
|---|---|
| 출력 | `outputs/report02_derived.json`, `outputs/rcs_anchor.json`, `outputs/sigma_anchor.json` |
| 소요 | 약 3분 (GPU 0장 — 이미 낸 σ 격자에 적합을 다시 건다) |
| 비고 | PO 면적분은 f² 로 커지는 정반사항을 담는다 — 그 기울기를 측정과 맞춘다 |

---


## 세 인자로 갈라 놓고 하나만 건드린다

게재된 표준트랙 논문의 분해를 그대로 쓴다(Zhang, IEEE JSAC 44:702, 2026 — 측정 적합 모델): **σ = A(f)·B₁(φ,θ)·B₂**. A(f) 는 주파수 의존성, B₁ 은 자세에 따른 모양, B₂ 는 자세 요동의 분포족이다.

**A(f) 의 기울기는 측정에서, A(f) 의 레벨과 B₁ 은 우리 계산에서 온다.** PO 면적분은 f² 로 커지는 정반사항을 담으므로 기울기가 기하에서 나오고, 그 하나를 측정에 맞춘다.


## 우리 기울기와 측정 기울기

기하에서 나온 우리 밴드 기울기는 -0.084 [^1] (mavic4pro [^2])~2.004 dB/GHz [^3] (phantom4 [^4]) 이고, 측정은 0.210 [^5] (Das, IEEE WCL 2026) 와 0.315 dB/GHz [^6] (Yuan, EuCAP 2025) 다.

⚠ **두 수는 창이 다르다** — 우리 값은 생산 세 밴드(1.843 [^19]~5.21 GHz [^20])에서 잰 기울기이고 측정 두 값은 1.8 [^21]~18.2 GHz [^22] 전대역 적합이다. 같은 기체를 같은 커널로 돌려도 창을 바꾸면 기울기가 달라진다 — 우리 Phantom 3 v2 메쉬에서 저대역 창 1.018 [^23] 대 전대역 0.519 dB/GHz [^24] 다.


## 앵커는 각 기체의 기울기를 어디로 옮기나

![report02_f6_band_slope](../outputs/figures/report02_f6_band_slope.png)

**그림 1.** 측정 앵커는 각 기체의 밴드 기울기를 어디로 옮기는가?


## 모드 선택 — 무엇을 옮기고 무엇을 대가로 내는가

| 모드 | 무엇을 옮기나 | 평균 레벨이동 [dB] | 대가 |
|---|---|---|---|
| slope_only (기본) | 주파수 의존성 A(f) 의 기울기만 | 0.00 | 없음 — 크기 가정 0개 |
| level_and_slope_L2 | 레벨 + 기울기 | -0.82 ~ +7.42 | 크기전이 L² 가정 (σ ∝ 투영면적) |
| level_and_slope_L4 | 레벨 + 기울기 | +1.93 ~ +16.92 | 크기전이 L⁴ 가정 (σ ∝ A²/λ²) · L² 와 최대 9.50 dB 차 (DJI S1000+) |

출처 [^25]

생산 기준은 `slope_only [^7]` 이고, 밴드별로 옮기는 양은 +2.70 [^8] (phantom4 [^9] @ LTE 1.843 GHz [^10]) ~ -2.41 dB [^11] (s1000plus [^12] @ WiFi 5.21 GHz [^13]) 다. 레벨이동 열은 기체 7 종 [^26] 의 세 밴드 평균 Δ 다.


## 왜 기울기만 받나

레벨까지 앵커에 맞추려면 크기전이 법칙을 하나 골라야 하고, 그 선택이 기체에 따라 최대 9.50 dB [^18] 를 정한다. 측정이 그 대가 없이 제약하는 양은 기울기뿐이므로 기울기만 받는다 [^27].

그 선택을 측정으로 닫는 자리가 [리포트 15 절 6 «캠페인이 결판내는 양은 절대값이 아니라 순위다»](15_measurement.ipynb) 이고, 절대 레벨을 처음으로 앵커하는 것은 [리포트 15 절 2 «구가 σ 를 절대량으로 만들고»](15_measurement.ipynb) 의 교정구다.


## 세 인자, 각각의 출처

| 인자 | 무엇 | 어디서 | 이 편의 근거 |
|---|---|---|---|
| A(f) 기울기 | 주파수 의존성 | **측정**(Das) | μ 기울기 0.21 dB/GHz [^5] |
| A(f) 레벨 | 절대 레벨 | **우리 PO 출력** | `slope_only [^7]` 의 레벨이동 절대 최대 0.00 dB [^16] |
| B₁(φ,θ) | 자세에 따른 모양 | **기하**(광선 가림 + PO) | 재보정 후 정규화 패턴 변화 1.9e-15 dB [^17] |
| B₂ | 자세 요동의 분포족 | 기하 | 문헌 적합 RMSE 2.41 dB [^28] 가 기준선 |


## 이 표가 서 있는 사슬 세대

⚠ **생산 앵커 원장과 위 문단의 우리 기울기는 σ 사슬의 서로 다른 세대다.** 생산 원장은 2026-07-30 07:16:08 [^29] 판 `rcs_anchor.json` 위에 서 있고, 위 문단은 디스크의 현재 판(2026-08-03 05:49:19 [^30])에서 다시 적합한 값이다. 겹치는 5 기체 [^31] 에서 밴드 기울기가 최대 1.396 dB/GHz [^32] 갈린다(mavic4pro [^33]).

⚠ **그리고 두 세대 모두 2026-08-04 [^34] 형상 정정 전 메쉬 위에 서 있다** — 이 축은 위 문단의 세대 축과 별개다. 앵커 5기체 중 Matrice 4E 가 그 정정을 받았다 [^35].

⚠ 셋째 축 — 두 세대 모두 2026-08-07 10:58:22 [^36] Γ(θ) 각도 모양(기본 켬) 이전 커널의 산출이기도 하다 — 전체 드론 σ 이동 +0.08 [^37] ~ +0.10 dB [^38] 다. 앵커를 갈아끼우려면 σ 사슬 전체를 형상 정정 + Γ(θ) 한 세대로 맞춰야 하고, 그것은 [리포트 13 «검출 결과»](13_results.ipynb) 전체가 먹는 값이다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 앵커 사슬(rcs_anchor → sigma_anchor)을 현재 메쉬·Γ(θ) 켠 커널로 다시 돌린다 | 밴드 기울기와 재보정 원장이 현재 기하 위에 선다 — 재생성 평균이 기체마다 -5.70 [^39] ~ +3.18 dB [^40] 로 갈린다 | `benchmark/rcs_anchor.py` → `src/sigma_anchor.py` |
| R90 사슬에 `modes.slope_only.delta_db` 를 적용한다 | 앵커 후 파형 순위가 확정된다 — 지금 적용하면 2 기체 [^41] 의 순위가 바뀐다 | `src/experiment_freespace_sigma.py` → [리포트 13 절 1 «레벨을 맞추려면 크기전이 법칙을 골라야 하므로…»](13_results.ipynb) |
| Matrice 4E · Mini 5 Pro 의 상대 레벨을 실측한다 | 크기전이 지수가 직접 고정되어 앵커 원장의 최대 항이 닫힌다 | [리포트 15 절 6 «캠페인이 결판내는 양은 절대값이 아니라 순위다»](15_measurement.ipynb) |
| 모서리 회절항(PTD)을 켜고 밴드 기울기를 다시 적합한다 | 면적분 밖의 항이 주파수 축을 얼마나 옮기는지가 확정된다 | `benchmark/rcs_anchor.py --ptd` |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 41개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report02_derived.json` | `band_slope.ours_min` | -0.08445 |
| [^2] | `outputs/report02_derived.json` | `band_slope.ours_min_drone` | mavic4pro |
| [^3] | `outputs/report02_derived.json` | `band_slope.ours_max` | 2.004 |
| [^4] | `outputs/report02_derived.json` | `band_slope.ours_max_drone` | phantom4 |
| [^5] | `outputs/rcs_anchor.json` | `literature.mu_eps.das_phantom3_mono.mu_a` | 0.21 |
| [^6] | `outputs/rcs_anchor.json` | `literature.mu_eps.yuan_phantom3_azplane.mu_a` | 0.315 |
| [^7] | `outputs/report02_derived.json` | `anchor_modes.production_mode` | slope_only |
| [^8] | `outputs/report02_derived.json` | `anchor.correction_max_db` | 2.698 |
| [^9] | `outputs/report02_derived.json` | `anchor.correction_max_drone` | phantom4 |
| [^10] | `outputs/report02_derived.json` | `anchor.correction_max_band` | LTE 1.843 GHz |
| [^11] | `outputs/report02_derived.json` | `anchor.correction_min_db` | -2.41 |
| [^12] | `outputs/report02_derived.json` | `anchor.correction_min_drone` | s1000plus |
| [^13] | `outputs/report02_derived.json` | `anchor.correction_min_band` | WiFi 5.21 GHz |
| [^14] | `outputs/report02_derived.json` | `anchor.slope_after_db_per_ghz` | 0.21 |
| [^15] | `outputs/report02_derived.json` | `anchor.slope_after_spread_db_per_ghz` | 1.887e-15 |
| [^16] | `outputs/report02_derived.json` | `anchor_modes.level_shift_abs_max_db` | 4.737e-15 |
| [^17] | `outputs/report02_derived.json` | `anchor.shape_invariance_max_abs_db` | 1.929e-15 |
| [^18] | `outputs/report02_derived.json` | `anchor_modes.size_law_spread_max_db` | 9.501 |
| [^19] | `outputs/report02_derived.json` | `bands_ghz.LTE` | 1.843 |
| [^20] | `outputs/report02_derived.json` | `bands_ghz.WiFi` | 5.21 |
| [^21] | `outputs/p3_validation_v2.json` | `slope.das_published.band[0]` | 1.8 |
| [^22] | `outputs/p3_validation_v2.json` | `slope.das_published.band[1]` | 18.2 |
| [^23] | `outputs/p3_validation_v2.json` | `slope.subband.ours.1.8-6.0 GHz.a` | 1.018 |
| [^24] | `outputs/p3_validation_v2.json` | `slope.ours_el0_full_band.a` | 0.5188 |
| [^25] | `outputs/report02_derived.json` | `anchor_modes.rows` | (3행 표) |
| [^26] | `outputs/report02_derived.json` | `anchor_modes.n_airframes` | 7 |
| [^27] | `outputs/report02_derived.json` | `anchor_modes.why_production` | 레벨까지 앵커에 맞추려면 크기전이 법칙을 하나 골라야 하고, 그 선택이 기체에 따라 최대 9.50… |
| [^28] | `outputs/rcs_anchor.json` | `literature.fit_rmse_db.AAV` | 2.414 |
| [^29] | `outputs/report02_derived.json` | `anchor.slope_ledger_source_generation` | 2026-07-30 07:16:08 |
| [^30] | `outputs/rcs_anchor.json` | `meta.generated` | 2026-08-03 05:49:19 |
| [^31] | `outputs/report02_derived.json` | `anchor.n_slope_crosschecked` | 5 |
| [^32] | `outputs/report02_derived.json` | `anchor.slope_ledger_gap_max_db_per_ghz` | 1.396 |
| [^33] | `outputs/report02_derived.json` | `anchor.slope_ledger_gap_max_drone` | mavic4pro |
| [^34] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^35] | `outputs/meshfix_attack.json` | `Q6_invalidated_outputs.critical[1]` | (3항목 묶음) |
| [^36] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^37] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.mini5pro.delta_db` | 0.08 |
| [^38] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.matrice4e.delta_db` | 0.1 |
| [^39] | `outputs/s2r_assets_verify.json` | `overstated[0].결과_표[0].mean_delta_db` | -5.7 |
| [^40] | `outputs/s2r_assets_verify.json` | `overstated[0].결과_표[2].mean_delta_db` | 3.18 |
| [^41] | `outputs/report02_derived.json` | `sigma_sens.anchor_not_applied_n_order_changed` | 2 |


---

## 절 2. 앵커가 통제한 항목과 남은 항목의 크기를 기체별 원장으로 적었다



> ### 한 일
> **앵커 기체와의 거리를 기체마다 판정하고, 앵커가 통제하지 못한 항목을 상태와 dB 크기로 함께 원장에 적었다.**

### 결과
1. 앵커 기체는 DJI Phantom 3 [^42] 한 대다 — 같은 급이면 `direct`(1 대 [^43]), 크기법칙으로 옮기면 `scaled`(5 대 [^44]), 위상 자체가 다르면 `not_comparable`(1 대 [^45]) 다.
2. 가장 큰 미통제 항은 **size transfer law [^46]** 9.50 dB [^47] 이고, 실제 적용한 보정 최대치 2.70 dB [^48] 보다 크다.
3. 그래서 커널은 그대로 두고 **원장으로만** 적용한다 — 검출 결과 편이 이 표를 함께 읽는다.
4. ⭐ `polarisation` 칸은 이제 크기를 갖는다 — 얇은 판 참값에서 폭 0.15 λ [^49] 일 때 두 편파가 11.56 dB [^50] 갈린다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 비교가능성 판정 | 앵커 기체와 같은 급이면 `direct`, 크기법칙으로 옮기면 `scaled`, 위상 자체가 다르면 `not_comparable` 로 적는다 |
| 미통제 항목 | 앵커가 닫지 못한 항목을 상태(RESOLVED/PARTIAL/UNRESOLVED)와 dB 크기로 함께 싣는다 — 크기를 못 붙인 항은 그렇게 적는다 |
| 왜 원장인가 | 가장 큰 미통제 항이 실제 보정보다 크다. 그래서 커널을 고치는 대신 원장으로 적고 결과 편이 나란히 읽게 한다 |

### 재현

```bash
PYTHONPATH=src python src/make_report02_target.py --derive-only
PYTHONPATH=src python src/build_part05_anchor.py
```

| | |
|---|---|
| 출력 | `outputs/sigma_anchor.json`, `outputs/report02_derived.json`, `outputs/lowfreq_anchor.json` |
| 소요 | 약 2분 (GPU 0장) |
| 비고 | 두 표 모두 생산 원장 세대이고 2026-08-04 [^51] 형상 정정 전 메쉬 위에 있다 |

---


## 기체마다 앵커와의 거리가 다르다

앵커 기체는 DJI Phantom 3 [^42] 한 대다. 같은 급이면 `direct`(1 대 [^43]), 크기법칙으로 옮기면 `scaled`(5 대 [^44]), 위상 자체가 다르면 `not_comparable`(1 대 [^45])로 적는다.


| 기체 | 대각 D [m] | D/D_ref | 로터 | 판정 | L²↔L⁴ 산포 [dB] |
|---|---|---|---|---|---|
| DJI Matrice 4E | 0.439 | 1.25 | 4 | scaled | 1.96 |
| DJI Mavic 4 Pro | 0.441 | 1.26 | 4 | scaled | 2.01 |
| DJI Mini 5 Pro | 0.275 | 0.79 | 4 | scaled | 2.09 |
| DJI Phantom 4 | 0.350 | 1.00 | 4 | direct | 0.00 |
| DJI S1000+ | 1.045 | 2.99 | 8 | not_comparable | 9.50 |
| Yuneec Typhoon H (H480) | 0.480 | 1.37 | 6 | scaled | 2.74 |
| Holybro X500 V2 | 0.500 | 1.43 | 4 | scaled | 3.10 |

출처 [^52]


## 앵커가 통제한 항목과 남은 항목의 크기

| 항목 | 상태 | 크기 [dB] |
|---|---|---|
| polarisation | UNRESOLVED | 미상 |
| statistic convention (Das mu) | RESOLVED_EMPIRICALLY | +0.93 |
| size transfer law | UNRESOLVED | +9.50 |
| single platform / single lab | UNRESOLVED | 미상 |
| elevation matching | PARTIAL | +0.73 |
| near-field vs far-field, environment | OK | +0.00 |

출처 [^53]

표의 `polarisation` 행 크기 칸은 원장이 null(미상)로 두고 있다 — 본문이 드는 11.56 dB [^50] 는 얇은 판 참값에서 온 값이고, 원장 반영은 다음 단계 표에 있다.


## 가장 큰 항이 실제 보정보다 크다

가장 큰 항은 **size transfer law [^46]** 9.50 dB [^47] 이고, 실제 적용한 보정 최대치 2.70 dB [^48] 보다 크다. 그래서 커널은 그대로 두고 **원장으로만** 적용한다 — [리포트 13 «검출 결과»](13_results.ipynb) 이 이 표를 함께 읽는다.

⭐ `polarisation` 칸은 이제 크기를 갖는다 — 얇은 판 참값에서 폭 0.15 λ [^49] 일 때 두 편파가 11.56 dB [^50] 갈리는데 우리 면적분은 편파를 가르는 대신 세기 하나만 내는 스칼라다([리포트 5 절 5 «PO 유효 무릎을 부품 폭으로 옮기면 어느 부…»](05_kernel.ipynb)). 그 낙차가 이 항의 크기이고, 부호는 [리포트 14 절 5 «교정된 절대 σ 를 만드는 조건은 여섯 항목이…»](14_robustness.ipynb) 의 VV/HH 측정이 정한다.

⚠ 두 표 모두 생산 원장 세대이고 2026-08-04 [^51] 형상 정정 전 메쉬 위에 있으며, 2026-08-07 10:58:22 [^54] Γ(θ) 각도 모양(기본 켬) 이전 커널의 산출이기도 하다 — 전체 드론 σ 이동 +0.08 [^55] ~ +0.10 dB [^56] 다 — Matrice 4E 행이 그 정정을 받은 기체다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| VV/HH 2편파를 잰다 | 원장의 편파 항이 크기에서 값으로 바뀐다 | [리포트 14 절 5 «교정된 절대 σ 를 만드는 조건은 여섯 항목이…»](14_robustness.ipynb) |
| Matrice 4E · Mini 5 Pro 의 상대 레벨을 실측한다 | 크기전이 지수가 직접 고정되어 원장의 최대 항이 닫힌다 | [리포트 15 절 6 «캠페인이 결판내는 양은 절대값이 아니라 순위다»](15_measurement.ipynb) |
| 정정된 메쉬로 앵커 원장 두 표를 다시 낸다 | 원장이 현재 형상 위에 선다 | [^57] |
| 고도 정합을 문헌 측정과 같은 격자로 좁힌다 | PARTIAL 로 남은 elevation 항이 닫힌다 | `benchmark/rcs_anchor.py` → 고도 격자 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 16개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^42] | `outputs/report02_derived.json` | `anchor.anchor_platform` | DJI Phantom 3 |
| [^43] | `outputs/report02_derived.json` | `anchor.n_direct` | 1 |
| [^44] | `outputs/report02_derived.json` | `anchor.n_scaled` | 5 |
| [^45] | `outputs/report02_derived.json` | `anchor.n_not_comparable` | 1 |
| [^46] | `outputs/report02_derived.json` | `anchor.largest_uncontrolled_term` | size transfer law |
| [^47] | `outputs/report02_derived.json` | `anchor.largest_uncontrolled_db` | 9.501 |
| [^48] | `outputs/report02_derived.json` | `anchor.correction_abs_max_db` | 2.698 |
| [^49] | `outputs/lowfreq_anchor.json` | `thin_plate.per_width.0.15.a_lam` | 0.15 |
| [^50] | `outputs/lowfreq_anchor.json` | `thin_plate.truth_2d_mom.0.15.tm_minus_te_db` | 11.56 |
| [^51] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^52] | `outputs/sigma_anchor.json` | `drones` | (7항목 묶음) |
| [^53] | `outputs/sigma_anchor.json` | `uncontrolled` | (6행 표) |
| [^54] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^55] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.mini5pro.delta_db` | 0.08 |
| [^56] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.matrice4e.delta_db` | 0.1 |
| [^57] | `outputs/meshfix_attack.json` | `recommended_gate_before_any_sigma_claim` | (6행 표) |


---

## 절 3. Phantom 3 를 문헌값을 보지 않고 내고 봉인을 풀었다



> ### 한 일
> **앵커 기체와 같은 기체를 문헌 상수를 한 번도 읽지 않은 경로로 돌리고 별도 스크립트가 봉인을 풀어 문헌과 맞댔다.**

### 결과
1. 같은 창 1.8 [^58]~18.2 GHz [^59] 에서 우리 커널로 돌렸다(`benchmark/p3_ours.py`, 14,542 s [^60]).
2. 우리 el=0 전대역 기울기는 0.420 [^61] ± 0.066 dB/GHz [^62] 이고, Das 공표 0.21 dB/GHz [^63] 대비 2.00 배 [^64] · 3.2 σ [^65] 다.
3. Yuan θ=90° 실측곡선 대비는 1.33 배 [^66] · 1.6 σ [^67] 이고 레벨은 -4.91 dB [^68] 다.
4. 우리 세 밴드가 전부 저주파 구간 안에 있고, 거기서 우리 μ(f) 는 1.56 [^69] (1.8–6.0 GHz)에서 0.20 dB/GHz [^70] (6.0–18.2 GHz)로 꺾인다.
5. 낙차의 방향은 얇은 판 참값이 정한다 — 전력을 만드는 쪽 기준이면 우리 PO 가 -4.02 [^71], 약한 쪽 기준이면 +7.53 dB [^72] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 눈감기 | 산출 과정은 문헌 상수를 한 번도 읽지 않았고 봉인은 별도 스크립트가 풀었다(`benchmark/p3_validation.py`) |
| 창 맞추기 | 적합 창은 Das Table III · Yuan §IV 와 같다 — 창을 바꾸면 같은 커널에서도 기울기가 달라지기 때문이다 |
| 무엇을 재나 | 절대 레벨과 주파수 의존 **두 스칼라**다. 각도패턴은 우리 기하에서 나온 그대로다 |
| 독립성 | Das 의 Phantom 3 행은 Yuan 원자료의 재분석이라 독립 2건이 아니다 |

### 재현

```bash
PYTHONPATH=src python benchmark/p3_ours.py
PYTHONPATH=src python benchmark/p3_validation.py
PYTHONPATH=src python src/build_part05_anchor.py
```

| | |
|---|---|
| 출력 | `outputs/p3_ours.json`, `outputs/p3_validation.json`, `outputs/lowfreq_anchor.json`, `outputs/lowfreq_attack.json` |
| 소요 | 약 4시간 (GPU 1장 — 전대역 σ 를 다시 낸다) |
| 비고 | 이 절의 표 네 행은 전부 v1 메쉬 산출이다 — v2 는 다음 편이 잇는다 |

---


## 무엇을 눈감고 냈나

앵커 기체와 같은 기체(DJI Phantom 3)를 같은 창 1.8 [^58]~18.2 GHz [^59] 에서 우리 커널로 돌렸다(`benchmark/p3_ours.py`, 14,542 s [^60]). 산출 과정은 문헌 상수를 한 번도 읽지 않았고 봉인은 별도 스크립트가 풀었다. 적합 창은 Das Table III · Yuan §IV 와 같다(일치 예 [^73]).

이 대조가 재는 것은 절대 레벨과 주파수 의존 두 스칼라이고, 각도패턴은 우리 기하에서 나온 그대로다. ⚠ 아래 표 네 행은 전부 **v1 메쉬** 산출이다 — 사진 실측으로 다시 지은 v2 메쉬의 같은 두 스칼라는 **절 4** «메쉬가 사는 축은 절대 크기가 아니라 각도 구…» 이 잇는다.


## 봉인을 풀고 맞댄 결과

| 대조 상대 | 고도 · 창 | 기울기 [dB/GHz] | 우리와의 거리 |
|---|---|---|---|
| 우리 el=0 전대역 (눈감기 산출) | el=0 · 1.8 [^58]–18.2 GHz [^59] | 0.420 [^61] ± 0.066 [^62] (양 끝점 제외 0.334 [^74]) | — |
| Das 공표 (IEEE WCL 2026) | 고도풀링 · 같은 창 | 0.21 [^63] | 2.00 배 [^64] · 3.2 σ [^65] |
| Yuan θ=90° 실측곡선 (EuCAP 2025) | 고도정합 · 같은 창 | 0.315 [^75] | 1.33 배 [^66] · 1.6 σ [^67] · 레벨 -4.91 dB [^68] |
| 부분대역 1.8–6.0 GHz (설명) | el=0 · 실측곡선 대조 | 우리 1.56 [^69] vs 측정 0.41 [^76] | 3.81 배 [^77] · 1.8 GHz 레벨 -7.97 dB [^78] |


## 정본은 전대역 대 전대역이다

부분대역 행은 우리 μ(f) 가 1.56 [^69] (1.8–6.0 GHz)에서 0.20 dB/GHz [^70] (6.0–18.2 GHz)로 꺾인다는 것을 보이는 설명이다.

⚠ **우리 세 밴드(LTE 1.843 [^79] · 5G 3.5 [^80] · WiFi 5.21 GHz [^81])가 전부 그 저주파 구간 안에 있고, 거기서 우리 σ 는 실측곡선보다 낮다.**


## 낙차의 방향은 어디까지 말할 수 있나

이 낙차의 방향은 결과 밖에 둔다 — 우리 PO 적분은 편파를 가르는 대신 세기 하나만 내는 스칼라라서, 참값이 편파에 따라 갈리는 구간에서는 한쪽 편파 기준으로 낮게 다른 쪽 기준으로 높게 나온다 [^82].

⭐ 다만 두 쪽의 크기가 서로 다르다 — 얇은 판 참값에서 **전력을 만드는 쪽**(두 편파 중 σ 가 11.56 dB [^83] 큰 쪽) 기준이면 우리 PO 는 -4.02 dB [^71] (음수 = 우리가 낮다), 약한 쪽 기준이면 +7.53 dB [^72] 다.

그래서 **우리 σ 가 낮게 나와 있을 개연성 쪽이 크고**, 그 방향이라면 검출 산출물은 보수적인(비관적인) 쪽으로 틀린 것이다. 레벨을 만드는 굵은 부품일수록 이 어긋남이 작아서 이 항이 설명하는 몫은 관측된 낙차보다 작다 [^84] — 그래서 방향은 여기까지, 개연성으로만 적는다. 금칙 세 가지는 [리포트 5 절 5 «PO 유효 무릎을 부품 폭으로 옮기면 어느 부…»](05_kernel.ipynb) 에 있다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| v2 메쉬로 같은 눈감기 대조를 다시 봉인해 돌린다 | 형상 개선이 두 스칼라를 어느 방향으로 옮기는지가 사전등록 아래에서 확정된다 | `benchmark/p3_ours.py` → **절 4** «메쉬가 사는 축은 절대 크기가 아니라 각도 구…» |
| 편파를 가르는 커널로 같은 창을 다시 낸다 | 낙차의 부호가 개연성에서 값으로 바뀐다 | [리포트 5 절 6 «커널이 아직 못 하는 것은 편파 분리·PTD·…»](05_kernel.ipynb) |
| 같은 잣대를 네 기체로 넓힌다 | 일치도가 한 기체의 우연인지 함대에서 서는 성질인지가 갈린다 | **절 5** «같은 잣대를 네 기체로 넓히면 판정이 NOT_…» |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 27개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^58] | `outputs/p3_validation.json` | `slope.das_published.band[0]` | 1.8 |
| [^59] | `outputs/p3_validation.json` | `slope.das_published.band[1]` | 18.2 |
| [^60] | `outputs/p3_ours.json` | `meta.runtime_s_total_process` | 1.454e+04 |
| [^61] | `outputs/p3_validation.json` | `slope.ours_el0_full_band.a` | 0.4198 |
| [^62] | `outputs/p3_validation.json` | `slope.ours_el0_full_band.se_a` | 0.06613 |
| [^63] | `outputs/p3_validation.json` | `slope.das_published.a` | 0.21 |
| [^64] | `outputs/p3_validation.json` | `slope.ratios.ours_over_das` | 1.999 |
| [^65] | `outputs/p3_validation.json` | `slope.significance.z_vs_das_0p21` | 3.172 |
| [^66] | `outputs/p3_validation.json` | `slope.ratios.ours_over_yuan_theta90` | 1.333 |
| [^67] | `outputs/p3_validation.json` | `slope.significance.z_vs_yuan_theta90_0p315` | 1.584 |
| [^68] | `outputs/p3_validation.json` | `residual.vs_yuan_theta90_measured_curve.mean_db` | -4.913 |
| [^69] | `outputs/p3_ours.json` | `subband_fits.by_aspect.el0.1.8-6.0 GHz.a` | 1.563 |
| [^70] | `outputs/p3_ours.json` | `subband_fits.by_aspect.el0.6.0-18.2 GHz.a` | 0.1986 |
| [^71] | `outputs/lowfreq_anchor.json` | `thin_plate.truth_2d_mom.0.15.po_minus_tm_db` | -4.022 |
| [^72] | `outputs/lowfreq_anchor.json` | `thin_plate.truth_2d_mom.0.15.po_minus_te_db` | 7.535 |
| [^73] | `outputs/p3_validation.json` | `window.same_window` | 예 |
| [^74] | `outputs/p3_validation.json` | `slope.ours_slope_robustness.drop_both_endpoints` | 0.3344 |
| [^75] | `outputs/p3_validation.json` | `slope.yuan_theta90_published.a` | 0.315 |
| [^76] | `outputs/p3_validation.json` | `our_operating_band.measured_theta90_curve_dense.a` | 0.4107 |
| [^77] | `outputs/p3_validation.json` | `our_operating_band.slope_ratio` | 3.807 |
| [^78] | `outputs/p3_validation.json` | `our_operating_band.level_error_db.at_1p8` | -7.968 |
| [^79] | `outputs/report02_derived.json` | `bands_ghz.LTE` | 1.843 |
| [^80] | `outputs/report02_derived.json` | `bands_ghz.5G` | 3.5 |
| [^81] | `outputs/report02_derived.json` | `bands_ghz.WiFi` | 5.21 |
| [^82] | `outputs/lowfreq_attack.json` | `what_survives_the_attack[4]` | ⑤ 스칼라 PO 의 원리적 한계는 확립됐다 — 참값이 0.15λ 에서 편파로 11.56 dB 갈라지… |
| [^83] | `outputs/lowfreq_anchor.json` | `thin_plate.truth_2d_mom.0.15.tm_minus_te_db` | 11.56 |
| [^84] | `outputs/lowfreq_attack.json` | `q4_A_and_B_are_not_exclusive.the_dB_decomposition_the_files_refuse_to_do.my_crude_budget.reading` | ⭐ B 는 부호가 맞고 크기가 **부족하다**. 이유는 구조적이다 — PO 오차가 가장 큰 특징(p… |


---

## 절 4. 메쉬가 사는 축은 절대 크기가 아니라 각도 구조다



> ### 한 일
> **Phantom 3 를 상자·정육면체·구로 바꿔 넣고 같은 눈금에서 다시 채점해 메쉬가 값어치를 내는 축을 갈랐다.**

### 결과
1. 레벨 축에서 우리 메쉬는 상자 계열을 전부 이긴다 — 가장 가까운 정육면체가 +6.22 [^85] 인데 우리는 -3.30 dB [^86] 다.
2. ⚠ 레벨 하나만 보면 부피를 맞게 고른 구가 우리보다 낫다 — 논문표기 상자와 부피가 같은 구가 +0.96 dB [^87] · rms 1.98 dB [^88] 다.
3. ⭐ 그 구의 부피를 무엇으로 잡는가가 자유 매개변수다 — 메쉬 부피와 같은 구는 -4.47 dB [^89] 로 우리보다 나쁘다.
4. 각도 축에서는 갈린다 — Das 잣대에서 우리는 +1.42 [^90], 상자 계열은 +3.98 [^91]~+4.52 dB [^92] 다.
5. v1→v2 메쉬는 레벨오차 -4.91 [^93] → -3.30 dB [^94], 기울기 0.420 [^95] → 0.519 dB/GHz [^96] 로 움직인다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 무엇을 갈아끼우나 | 같은 자리·같은 창에서 Phantom 3 를 논문표기 상자·정육면체·구로 바꿔 넣고 다시 채점한다 |
| 레벨의 잣대 | Yuan θ90 복원 실측곡선 기준 밴드평균 잔차 [^97] |
| 각도의 잣대 | ⚠ ε(방위 산포)만은 잣대가 다르다 — 비교 상대가 Das Table III 가 준 값이다. 레벨은 Yuan 잣대, ε 는 Das 잣대다 |
| 메쉬 세대 | 이 편의 «우리» 는 v2 메쉬(사진 실측으로 다시 지은 판)이고, 눈감기 대조의 «우리» 는 v1 이다 |

### 재현

```bash
PYTHONPATH=src python benchmark/p3_validation_v2.py
PYTHONPATH=src python src/build_part05_anchor.py
```

| | |
|---|---|
| 출력 | `outputs/p3_validation_v2.json`, `outputs/das_fleet_validation.json` |
| 소요 | 약 2시간 (GPU 1장 — 대조군 형상마다 σ 를 다시 낸다) |
| 비고 | 구는 방위에 따라 σ 가 그대로라 ε 를 0 으로 낸다 |

---


## 무엇을 갈아끼웠나

**레벨을** 같은 눈금(Yuan θ90 복원 실측곡선 기준 밴드평균 잔차 [^97])으로 놓고 Phantom 3 를 상자·정육면체·구로 바꿔 넣어 다시 채점했다. 우리 메쉬는 **상자 계열을 전부 이긴다** — 가장 가까운 정육면체가 +6.22 [^85] 인데 우리는 -3.30 dB [^86] 다.

⚠ 그런데 **레벨 하나만 보면 부피를 맞게 고른 구가 우리보다 낫다** — 논문표기 상자와 부피가 같은 구가 +0.96 dB [^87] · 주파수 전체에 걸친 평균 오차 크기(rms) 1.98 dB [^88] 다. ⭐ 다만 **그 구의 부피를 무엇으로 잡는가가 자유 매개변수**(결과를 보고 골라 넣을 수 있는 값)다 — 메쉬 부피와 같은 구는 -4.47 dB [^89] 로 우리보다 나쁘다.


## v1 과 v2 는 다른 메쉬다

⭐ **이 편의 «우리» 는 v2 메쉬**(사진 실측으로 다시 지은 판)이고, **절 3** «Phantom 3 를 문헌값을 보지 않고 내고…» 표의 «우리» 는 v1 메쉬다. 같은 Yuan θ90 곡선 잣대에서 v1→v2 는 레벨오차 -4.91 [^93] → -3.30 dB [^94], 기울기 0.420 [^95] → 0.519 dB/GHz [^96] (촘촘하게 되살린 실측곡선 0.316 dB/GHz [^98])로 움직인다 — 레벨은 가까워지고 기울기는 멀어졌다.

형상은 삼각형 28,160 [^99]→28,366 [^100] · 실루엣 IoU 가 자기복제 상한 대비 77.9 [^101]→86.9% [^102] 다.


## 각도 축에서는 갈린다

구는 방위에 따라 σ 가 그대로라 방위 산포 ε(같은 기체를 여러 방위에서 볼 때 σ 가 얼마나 흩어지는지, dB)를 0.00 dB [^103] 로 낸다.

⚠ **ε 만은 잣대가 다르다** — 비교 상대가 위 레벨이 쓴 Yuan 곡선이 아니라 Das Table III 가 준 5.46 dB [^104] 다. 그 Das 잣대에서 우리는 +1.42 [^90], 상자 계열은 +3.98 [^91]~+4.52 dB [^92] 다.

**그래서 메쉬가 값어치를 내는 축은 σ 의 절대 크기가 아니라 각도에 따른 구조다** — 절대 레벨은 [리포트 15 절 2 «구가 σ 를 절대량으로 만들고»](15_measurement.ipynb) 의 교정구가 측정으로 앵커한다.


## 레벨에서 구가 이긴 것은 잣대가 아니라 어느 구를 넣었나가 갈랐다

⭐ 메쉬 부피로 잡은 같은 반지름의 구를 레벨까지 Das Table III 잣대로 다시 채점해도 같은 Phantom 3 에서 우리(-1.47 [^105])가 구(-2.59 dB [^106])를 이긴다. 그쪽 표는 구의 σ 를 단면적 πa² 로 굳힌 근사로, 이 표는 정확해(Mie)로 낸다 — 반지름은 같다.

즉 **두 잣대 모두에서 지는 것은 메쉬 부피로 잡은 구**이고, 레벨에서 우리를 앞선 것은 반지름을 그보다 키운 논문표기 상자부피 구 하나다. 두 결과는 «어느 부피를 골랐나» 와 함께 읽는다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| Phantom 3 구 대조군을 방위 산포 ε 축까지 포함해 다섯 기체로 넓힌다 | 형상 모형이 각도 구조를 사는 폭이 기체마다 확정된다 | `benchmark/p3_validation_v2.py` 확장 |
| 교정구를 표적과 같은 자리에서 함께 잰다 | 레벨 축이 측정으로 닫힌다 — 지금은 어느 부피를 골랐나가 결과를 정한다 | [리포트 15 절 2 «구가 σ 를 절대량으로 만들고»](15_measurement.ipynb) |
| 같은 대조군을 함대 사전등록 채점으로 넓힌다 | 형상 증거의 유무가 일치도를 가르는지가 네 기체에서 확정된다 | **절 5** «같은 잣대를 네 기체로 넓히면 판정이 NOT_…» |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 22개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^85] | `outputs/p3_validation_v2.json` | `controls.table.cube_vol_v2.level_err_db` | 6.222 |
| [^86] | `outputs/p3_validation_v2.json` | `controls.table.ours_phantom3_mesh_v2.level_err_db` | -3.304 |
| [^87] | `outputs/p3_validation_v2.json` | `controls.table.sphere_eqvol_paperbox.level_err_db` | 0.9553 |
| [^88] | `outputs/p3_validation_v2.json` | `controls.table.sphere_eqvol_paperbox.rms_db` | 1.982 |
| [^89] | `outputs/p3_validation_v2.json` | `controls.table.sphere_vol_v2.level_err_db` | -4.465 |
| [^90] | `outputs/p3_validation_v2.json` | `controls.table.ours_phantom3_mesh_v2.eps_err_vs_das_db` | 1.417 |
| [^91] | `outputs/p3_validation_v2.json` | `controls.table.cube_vol_v2.eps_err_vs_das_db` | 3.977 |
| [^92] | `outputs/p3_validation_v2.json` | `controls.table.box_paper.eps_err_vs_das_db` | 4.516 |
| [^93] | `outputs/p3_validation_v2.json` | `v1_vs_v2.level_db.v1` | -4.913 |
| [^94] | `outputs/p3_validation_v2.json` | `v1_vs_v2.level_db.v2` | -3.304 |
| [^95] | `outputs/p3_validation_v2.json` | `v1_vs_v2.slope_db_per_ghz.v1` | 0.4198 |
| [^96] | `outputs/p3_validation_v2.json` | `v1_vs_v2.slope_db_per_ghz.v2` | 0.5188 |
| [^97] | `outputs/p3_validation_v2.json` | `controls.scoring` | Yuan θ90 복원 실측곡선 기준 밴드평균 잔차 |
| [^98] | `outputs/p3_validation_v2.json` | `v1_vs_v2.slope_db_per_ghz.measured_comparand_dense` | 0.3164 |
| [^99] | `outputs/p3_validation_v2.json` | `v1_vs_v2.mesh.v1_n_tri` | 28160 |
| [^100] | `outputs/p3_validation_v2.json` | `v1_vs_v2.mesh.v2_n_tri` | 28366 |
| [^101] | `outputs/p3_validation_v2.json` | `v1_vs_v2.mesh.silhouette_iou_pct_of_ceiling.old` | 77.9 |
| [^102] | `outputs/p3_validation_v2.json` | `v1_vs_v2.mesh.silhouette_iou_pct_of_ceiling.new` | 86.9 |
| [^103] | `outputs/p3_validation_v2.json` | `controls.table.sphere_eqvol_paperbox.eps_mean_db` | 0 |
| [^104] | `outputs/p3_validation_v2.json` | `v1_vs_v2.eps_db.das_mean` | 5.46 |
| [^105] | `outputs/das_fleet_validation.json` | `box_control.phantom3.ours_DL_db` | -1.47 |
| [^106] | `outputs/das_fleet_validation.json` | `box_control.phantom3.controls.sphere_eqvol.DL_db` | -2.592 |


---

## 절 5. 같은 잣대를 네 기체로 넓히면 판정이 NOT_VALIDATED 로 갈린다



> ### 한 일
> **계산 전에 봉인한 합격규칙을 그대로 걸어 Das Table III 의 네 기체 · 바이스태틱각 일곱 개를 전수 채점했다.**

### 결과
1. 판정은 **NOT_VALIDATED (P3 산포) [^107]** 다 — 결정권을 준 항목은 레벨오차 산포가 6.0 dB 이하일 것이고, 실제 산포는 16.27 dB [^108] 다.
2. 봉인이 예측한 산포는 1.0 dB [^109] 였다 — 예측이 반증됐다.
3. ⭐ 갈린 축은 대역도 전기적 크기도 기체 크기도 아니고 **그 기체 자체의 형상 증거가 있는가** 다 — 증거가 있는 Mini 2 -0.51 [^110] · Phantom 3 -1.47 [^111] 와 얇은 Phantom 2 -7.19 [^112] · M350 RTK +9.08 dB [^113] 사이에 5.72 dB [^114] 의 빈 구간이 있다.
4. 예측이 반증됐고 일치도가 메쉬 구속도를 따라간다는 두 사실이 함께 서 있다 — 그것은 결과에 맞춰 맞춘 흔적의 정반대 서명이다 [^115].


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 사전등록 | 합격규칙은 **계산 전에** 봉인했고(`outputs/das_fleet_prereg.json`) 그대로 썼다 |
| 결정 항목 | ⭐ max(DL(0)) − min(DL(0)) ≤ 6.0 dB — 오차가 기체마다 흩어진 게 아니라 **공통 오프셋**일 것 |
| DL(0) 이란 | 바이스태틱각 0°(송신기와 수신기가 같은 자리)에서 측정 대비 우리 σ 가 몇 dB 높거나 낮은가다 — 양수면 우리가 더 밝게 냈다 |
| P3 이란 | 판정 문자열의 P3 은 봉인한 합격조건 네 개 중 **세 번째**(레벨오차의 산포)를 가리키는 이름이고 기체 Phantom 3 를 뜻하지 않는다 |

### 재현

```bash
PYTHONPATH=src python benchmark/das_fleet_validation.py
PYTHONPATH=src python src/build_part05_anchor.py
```

| | |
|---|---|
| 출력 | `outputs/das_fleet_validation.json`, `outputs/das_fleet_prereg.json`, `outputs/das_fleet_attack.json` |
| 소요 | 약 3시간 (GPU 1장 — 네 기체를 문헌 격자에서 다시 낸다) |
| 비고 | 봉인 파일은 채점 전 커밋에 그대로 있다 |

---


## 봉인하고 채점했다

Das Table III 는 네 기체를 바이스태틱각 일곱 개로 준다 — 그 칸을 전부 채점했다([^116]). 합격규칙은 **계산 전에** 봉인했고(`outputs/das_fleet_prereg.json`) 그대로 썼다.

결정권을 준 항목은 「⭐ max(DL(0)) − min(DL(0)) ≤ 6.0 dB — 오차가 기체마다 흩어진 게 아니라 **공통 오프셋**일 것 [^117]」 이다.


## 판정

판정은 **NOT_VALIDATED (P3 산포) [^107]** 다 — 실제 산포가 16.27 dB [^108] 이고 봉인이 예측한 산포는 1.0 dB [^109] 였다.

⚠ 용어 두 개를 푼다. 판정 문자열의 **P3 은 봉인한 합격조건 네 개 중 세 번째**(레벨오차 산포)를 가리키는 이름이고 기체 Phantom 3 를 뜻하지 않는다. 그리고 **레벨오차 DL(0)** 는 바이스태틱각 0°(송신기와 수신기가 같은 자리에 있는 배치)에서 측정 대비 우리 σ 가 몇 dB 높거나 낮은가다 — 양수면 우리가 더 밝게 냈다는 뜻이다.


## 갈린 축은 형상 증거의 유무다

⭐ 갈린 축은 대역도 전기적 크기도 기체 크기도 아니고 **그 기체 자체의 형상 증거가 있는가** 다 — 형상 증거가 있는 Mini 2 -0.51 [^110] · Phantom 3 -1.47 [^111] 와 증거가 얇은 Phantom 2 -7.19 [^112] · M350 RTK +9.08 dB [^113] 사이에 5.72 dB [^114] 의 빈 구간이 있다.

예측이 반증됐고 일치도가 메쉬 구속도를 따라간다는 두 사실이 함께 서 있고, 그것은 결과에 맞춰 맞춘 흔적의 정반대 서명이다 [^115].

⚠ M350 RTK 값은 계산이 다 끝나기 전의 스냅샷에서 집계됐고, Mini 2 행은 2026-08-04 [^118] 형상 정정 전 메쉬에서 나온 값이다 — 둘 다 재집계를 기다린다 [^119] · [^120]. 재집계에는 Γ(θ) 커널(2026-08-07 10:58:22 [^121] 기본 켬) 반영도 함께 든다 — 판정(NOT_VALIDATED) 자체는 이 크기의 이동에 강건하다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| M350 RTK 를 계산이 끝난 격자에서 다시 집계한다 | 네 기체 중 한 행의 값이 확정되어 산포가 다시 계산된다 | [^119] |
| Mini 2 를 정정된 메쉬로 다시 낸다 | 형상 증거 축이 정정 후에도 서는지가 확정된다 | [^120] |
| 형상 증거가 있는 기체를 실측으로 늘린다 | 형상 증거 ↔ 일치도의 관계가 우리 1차 표적 두 대에서 확정된다 | [리포트 15 절 6 «캠페인이 결판내는 양은 절대값이 아니라 순위다»](15_measurement.ipynb) |
| 다음 라운드의 합격규칙을 같은 방식으로 미리 봉인한다 | 판정이 결과를 보고 조정되지 않았다는 것이 다음 라운드에서도 유지된다 | [리포트 1 절 3 «주장마다 판정 범위를 결판·사슬확인·캠페인 밖…»](01_map.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 15개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^107] | `outputs/das_fleet_validation.json` | `prereg_judgement.verdict` | NOT_VALIDATED (P3 산포) |
| [^108] | `outputs/das_fleet_validation.json` | `prereg_judgement.P3_spread_db` | 16.27 |
| [^109] | `outputs/das_fleet_validation.json` | `prereg_judgement.spread_predicted_db` | 1 |
| [^110] | `outputs/das_fleet_validation.json` | `prereg_judgement.DL0_db.mini2` | -0.5077 |
| [^111] | `outputs/das_fleet_validation.json` | `prereg_judgement.DL0_db.phantom3` | -1.47 |
| [^112] | `outputs/das_fleet_validation.json` | `prereg_judgement.DL0_db.phantom2` | -7.192 |
| [^113] | `outputs/das_fleet_validation.json` | `prereg_judgement.DL0_db.m350rtk` | 9.076 |
| [^114] | `outputs/das_fleet_validation.json` | `degrees_of_freedom.evidence_split.gap_db` | 5.722 |
| [^115] | `outputs/das_fleet_attack.json` | `Q5_tuned_evidence` | (4항목 묶음) |
| [^116] | `outputs/das_fleet_validation.json` | `table_28` | (28행 표) |
| [^117] | `outputs/das_fleet_prereg.json` | `pass_rule.primary_gate_level_at_theta_b_0.P3_spread` | ⭐ max(DL(0)) - min(DL(0)) <= 6.0 dB — 오차가 기체마다 흩어진 게 아니… |
| [^118] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^119] | `outputs/das_fleet_attack.json` | `overall.required_before_citing[0]` | m350rtk 계산을 끝내고 das_fleet_ours.json 을 **재집계**한 뒤 +9.08… |
| [^120] | `outputs/meshfix_attack.json` | `Q6_invalidated_outputs.critical[0]` | (5항목 묶음) |
| [^121] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |


---

## 절 6. 공통모드 σ 오차는 파형 순위를 안 건드리고, 차분 오차가 뒤집는다



> ### 한 일
> **σ 오차를 공통모드와 차분 두 갈래로 나눠 검출거리 R90 순위에 넣고 각각이 순위를 언제 뒤집는지 쟀다.**

### 결과
1. 기체 5 [^122] × 밴드 3 [^123] = 15 셀 [^124] 에서 쟀다.
2. 공통모드에서 순위는 -10 [^125]~+10 dB [^126] 전 구간 15 셀 [^124] 에서 그대로다 — 움직이는 것은 절대거리뿐이다.
3. 그 기울기가 σ 1 dB 당 0.246 dB [^127] (0.223 [^128]~0.256 [^129]), 즉 σ ±10 dB 에서 거리 -43 [^130] %~+76% [^131] 다.
4. 자세평균 σ 로 인용하면 다섯 기체가 한 순위(LTE > 5G > WiFi [^132])에 합의하고, 단일자세에서는 순위 3 종 [^133] 이 나온다.
5. 거기에 측정 기울기를 얹으면 최악 뒤집힘 문턱이 0.09 [^134] → 3.72 dB [^135] 로 올라간다(+3.63 dB [^136]).


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 두 갈래 | **공통모드**는 한 기체의 세 밴드를 같은 dB 로 옮기고(절대 레벨 오차의 모양), **차분**은 밴드마다 다르게 옮긴다(기울기 오차의 모양) |
| 무엇에 넣나 | 검출거리 R90 — 검출확률 90 % 문턱을 마지막으로 넘는 거리다 (`benchmark/sigma_sensitivity.py`) |
| 무엇을 보나 | 절대거리가 아니라 **세 파형의 순위**가 언제 뒤집히는지를 본다 |
| 왜 이 축인가 | 차분오차가 순위를 정하는 축이고, 그래서 앵커가 잡는 축이 정확히 이것이다 |

### 재현

```bash
PYTHONPATH=src python benchmark/sigma_sensitivity.py
PYTHONPATH=src python src/build_part05_anchor.py
```

| | |
|---|---|
| 출력 | `outputs/sigma_sensitivity.json`, `outputs/report02_derived.json` |
| 소요 | 약 20분 (GPU 1장 — 검출 사슬을 오차마다 다시 푼다) |
| 비고 | 논문이 절대 σ 에 기대는 곳은 공통모드 문단에서 끝난다 |

---


## 오차를 두 갈래로 나눈다

σ 오차를 두 갈래로 나눠 검출거리 R90(검출확률 90 % 문턱을 마지막으로 넘는 거리) 순위에 넣었다(`benchmark/sigma_sensitivity.py`, 기체 5 [^122] × 밴드 3 [^123] = 15 셀 [^124]).

**공통모드**는 한 기체의 세 밴드를 같은 dB 로 옮긴다 — 절대 레벨 오차의 모양이다. **차분**은 밴드마다 다르게 옮긴다 — 기울기 오차의 모양이다.


## 공통모드 — 절대거리만 움직인다

공통모드에서 순위는 -10 [^125]~+10 dB [^126] 전 구간 15 셀 [^124] 에서 그대로다 — 움직이는 것은 절대거리뿐이고 그 기울기가 σ 1 dB 당 0.246 dB [^127] (0.223 [^128]~0.256 [^129]), 즉 σ ±10 dB 에서 거리 -43 [^130] %~+76% [^131] 다.

**논문이 절대 σ 에 기대는 곳은 여기서 끝난다.**


## 차분 — 순위를 정하는 축

차분오차가 순위를 정하는 축이고, 그래서 **절 1** «σ = A(f)·B₁·B₂ 에서 A(f) 의…» 의 앵커가 잡는 축이 정확히 이것이다.

자세평균 σ 로 인용하면 다섯 기체가 한 순위(LTE > 5G > WiFi [^132])에 합의하고(단일자세에서는 순위 3 종 [^133]), 거기에 측정 기울기를 얹으면 최악 뒤집힘 문턱이 0.09 [^134] → 3.72 dB [^135] 로 올라간다(+3.63 dB [^136]).


## σ 오차가 어느 축으로 커질 때 순위가 뒤집히는가

![report02_f7_sigma_sensitivity](../outputs/figures/report02_f7_sigma_sensitivity.png)

**그림 1.** σ 오차가 어느 축으로 얼마나 커질 때 세 파형의 순위가 뒤집히는가?


| 기체 | 최대 치수 [m] | D/λ @LTE | 뒤집힘 문턱 · 단일자세 [dB] | 뒤집힘 문턱 · 자세평균 [dB] | 밴드간 σ 산포 [dB] | P(순위 보존) @1 dB |
|---|---|---|---|---|---|---|
| DJI Mini 5 Pro ⭐ | 0.378 | 2.32 | 7.42 | 2.95 | 2.2 | 0.992 |
| DJI Phantom 4 | 0.471 | 2.89 | 5.03 | 2.20 | 14.8 | 0.947 |
| DJI Mavic 4 Pro | 0.556 | 3.42 | 1.84 | 0.09 | 18.7 | 0.745 |
| DJI Matrice 4E ⭐ | 0.587 | 3.61 | 0.61 | 1.30 | 5.5 | 0.583 |
| DJI S1000+ | 1.348 | 8.29 | 0.79 | 0.80 | 10.4 | 0.419 |

출처 [^137]


## 이 표가 사는 주장

뒤집힘 문턱은 기체 크기가 아니라 **로브 산포**가 정한다 — 표에서 최대 치수 열과 뒤집힘 문턱 열이 같은 방향으로 가지 않는다. 밴드간 σ 산포 열이 그 순서를 만든다.

⚠ Matrice 4E 행은 2026-08-04 [^138] 형상 정정 전 값이고, 표 전체가 2026-08-07 10:58:22 [^139] Γ(θ) 각도 모양(기본 켬) 이전 커널의 산출이기도 하다 — 전체 드론 σ 이동 +0.08 [^140] ~ +0.10 dB [^141] 다 — 이 표를 낸 `outputs/sigma_sensitivity.json` 가 그 기체의 형상을 먹는다 [^142]. 재생성 시 0.1 dB 급 문턱 값은 그 이동만으로도 움직인다 — 한 기체 값이 움직여도 이 표가 사는 주장은 그대로 선다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| Matrice 4E 를 정정된 메쉬로 다시 넣어 표를 갱신한다 | 한 행의 값이 현재 형상 위에 선다 | [^142] |
| 공통모드 기울기를 검출 결과 편의 σ-free 축과 나란히 읽는다 | σ 를 곱하기 전에 이미 순위를 정하는 축이 무엇인지가 확정된다 | [리포트 14 절 1 «σ 를 곱하기 전에 이미 세 파형의 순서를 정…»](14_robustness.ipynb) |
| 자세평균 인용 규약을 σ 인용 단위로 굳힌다 | 어느 통계로 인용해야 순위가 서는지가 규약으로 확정된다 | [리포트 13 절 3 «그 순위는 자세평균이면 σ 오차 아래에서 하나…»](13_results.ipynb) |
| 차분오차의 실제 크기를 실측으로 잰다 | 뒤집힘 문턱과 실제 오차의 거리가 확정된다 | [리포트 15 절 7 «기울기 판정의 문턱은 세션간 진폭 재현성이고»](15_measurement.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 21개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^122] | `outputs/report02_derived.json` | `sigma_sens.n_airframes` | 5 |
| [^123] | `outputs/report02_derived.json` | `sigma_sens.n_bands` | 3 |
| [^124] | `outputs/report02_derived.json` | `sigma_sens.n_cells` | 15 |
| [^125] | `outputs/report02_derived.json` | `sigma_sens.offset_min_db` | -10 |
| [^126] | `outputs/report02_derived.json` | `sigma_sens.offset_max_db` | 10 |
| [^127] | `outputs/report02_derived.json` | `sigma_sens.slope_mean_db_per_db` | 0.2462 |
| [^128] | `outputs/report02_derived.json` | `sigma_sens.slope_min_db_per_db` | 0.2227 |
| [^129] | `outputs/report02_derived.json` | `sigma_sens.slope_max_db_per_db` | 0.2561 |
| [^130] | `outputs/report02_derived.json` | `sigma_sens.range_at_minus10_pct` | -43.31 |
| [^131] | `outputs/report02_derived.json` | `sigma_sens.range_at_plus10_pct` | 76.39 |
| [^132] | `outputs/report02_derived.json` | `sigma_sens.aspect_avg_order` | LTE > 5G > WiFi |
| [^133] | `outputs/report02_derived.json` | `sigma_sens.single_aspect_n_orders` | 3 |
| [^134] | `outputs/report02_derived.json` | `sigma_sens.worst_flip_aspect_avg_db` | 0.08856 |
| [^135] | `outputs/report02_derived.json` | `sigma_sens.worst_flip_anchored_db` | 3.72 |
| [^136] | `outputs/report02_derived.json` | `sigma_sens.anchor_gain_db` | 3.631 |
| [^137] | `outputs/report02_derived.json` | `sigma_sens.rows` | (5행 표) |
| [^138] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^139] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^140] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.mini5pro.delta_db` | 0.08 |
| [^141] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.matrice4e.delta_db` | 0.1 |
| [^142] | `outputs/meshfix_attack.json` | `Q6_invalidated_outputs.critical[3]` | (4항목 묶음) |
